# S04 - Explainability avec Grad-CAM

Objectif:

1. Charger le modele CNN entraine.
2. Predire des images IRM du test.
3. Visualiser les zones qui influencent la prediction avec Grad-CAM.
4. Comparer prediction correcte et incorrecte.
5. Fournir une interpretation medicale prudente.

Pourquoi Grad-CAM ?

Dans un projet medical, il ne suffit pas d'avoir une accuracy elevee. Il faut aussi comprendre quelles zones de l'image influencent le modele. Grad-CAM produit une heatmap qui indique les regions importantes pour la decision du CNN.


## 1. Imports et chemins


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFilter

import tensorflow as tf
from tensorflow import keras

from sklearn.metrics import classification_report, confusion_matrix

plt.style.use("default")
sns.set_theme(style="whitegrid")

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Alzheimer_CV_Project")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODELS_DIR = PROJECT_DIR / "models"

MODEL_PATH = MODELS_DIR / "balanced_alzheimer_cnn.keras"
CLASS_MAPPING_PATH = MODELS_DIR / "class_mapping.json"

IMG_SIZE = (160, 160)
RANDOM_STATE = 42

print("Model path:", MODEL_PATH)
print("Model exists:", MODEL_PATH.exists())
print("Class mapping exists:", CLASS_MAPPING_PATH.exists())


### Interpretation

Cette cellule verifie que le modele entraine existe bien. Grad-CAM utilise les couches convolutionnelles du CNN sauvegarde.


## 2. Charger le modele et les classes


In [ ]:
model = keras.models.load_model(MODEL_PATH)

with open(CLASS_MAPPING_PATH, "r", encoding="utf-8") as f:
    mapping = json.load(f)

class_to_index = mapping["class_to_index"]
index_to_class = {int(k): v for k, v in mapping["index_to_class"].items()}
classes = [index_to_class[i] for i in sorted(index_to_class)]

print("Classes:", classes)
model.summary()


### Interpretation

On recharge le modele final et l'ordre des classes. C'est important car la sortie du reseau est un vecteur de probabilites, et chaque position correspond a une classe precise.


## 3. Charger les donnees test


In [ ]:
test_df = pd.read_csv(PROCESSED_DIR / "test_split.csv")

test_augmented_path = PROCESSED_DIR / "test_augmented_robustness.csv"
if test_augmented_path.exists():
    test_augmented_df = pd.read_csv(test_augmented_path)
else:
    test_augmented_df = None

print("Test original:", len(test_df))
if test_augmented_df is not None:
    print("Test augmente robustesse:", len(test_augmented_df))

display(test_df["label"].value_counts().loc[classes])


### Interpretation

On utilise d'abord le test original, car c'est l'evaluation principale. Le test augmente peut etre utilise ensuite pour verifier si les zones Grad-CAM restent coherentes sur des images legerement modifiees.


## 4. Preprocessing identique a l'entrainement


In [ ]:
def crop_non_black_region_pil(img, threshold=8, padding=6):
    arr = np.array(img)
    mask = arr > threshold
    if mask.sum() == 0:
        return img

    ys, xs = np.where(mask)
    y0, y1 = ys.min(), ys.max()
    x0, x1 = xs.min(), xs.max()

    y0 = max(0, y0 - padding)
    y1 = min(arr.shape[0] - 1, y1 + padding)
    x0 = max(0, x0 - padding)
    x1 = min(arr.shape[1] - 1, x1 + padding)
    return img.crop((x0, y0, x1 + 1, y1 + 1))


def load_preprocess_image(path, img_size=IMG_SIZE):
    img = Image.open(path).convert("L")
    img = crop_non_black_region_pil(img)
    img = img.filter(ImageFilter.MedianFilter(size=3))
    img = img.resize(img_size)
    arr = np.array(img, dtype=np.float32) / 255.0
    arr = np.expand_dims(arr, axis=-1)
    return arr

print("Preprocessing ready.")


### Interpretation

Grad-CAM doit utiliser exactement le meme preprocessing que le modele. Sinon, l'image expliquee ne correspond pas a l'image vue par le CNN.


## 5. Identifier la derniere couche convolutionnelle


In [ ]:
def find_last_conv_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer, keras.layers.Conv2D):
            return layer.name
    raise ValueError("Aucune couche Conv2D trouvee dans le modele.")

last_conv_layer_name = find_last_conv_layer(model)
print("Derniere couche convolutionnelle:", last_conv_layer_name)


### Interpretation

Grad-CAM utilise la derniere couche convolutionnelle, car elle contient des informations visuelles de haut niveau: formes, structures et regions discriminantes.


## 6. Fonction Grad-CAM


In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output],
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_heatmap_on_gray(img_array, heatmap, alpha=0.38):
    img = img_array.squeeze()
    heatmap_resized = Image.fromarray(np.uint8(255 * heatmap)).resize(img.shape[::-1])
    heatmap_resized = np.array(heatmap_resized) / 255.0

    cmap = plt.cm.jet(heatmap_resized)[..., :3]
    gray_rgb = np.repeat(img[..., np.newaxis], 3, axis=-1)
    overlay = (1 - alpha) * gray_rgb + alpha * cmap
    overlay = np.clip(overlay, 0, 1)
    return overlay, heatmap_resized

print("Grad-CAM functions ready.")


### Interpretation

La heatmap Grad-CAM indique les zones qui augmentent la probabilite de la classe predite. Les zones chaudes ne sont pas une segmentation medicale, mais une explication visuelle de la decision du modele.


## 7. Prediction + Grad-CAM sur une image


In [ ]:
def predict_and_gradcam(image_path, true_label=None, show=True):
    img_array = load_preprocess_image(image_path)
    input_batch = img_array[np.newaxis, ...]

    proba = model.predict(input_batch, verbose=0)[0]
    pred_idx = int(np.argmax(proba))
    pred_label = index_to_class[pred_idx]
    confidence = float(proba[pred_idx])

    heatmap = make_gradcam_heatmap(input_batch, model, last_conv_layer_name, pred_index=pred_idx)
    overlay, heatmap_resized = overlay_heatmap_on_gray(img_array, heatmap)

    result_df = pd.DataFrame({
        "class": classes,
        "probability": proba,
    }).sort_values("probability", ascending=False)

    if show:
        plt.figure(figsize=(14, 4))

        plt.subplot(1, 4, 1)
        plt.imshow(img_array.squeeze(), cmap="gray")
        plt.title("Image preprocess")
        plt.axis("off")

        plt.subplot(1, 4, 2)
        plt.imshow(heatmap_resized, cmap="jet")
        plt.title("Heatmap Grad-CAM")
        plt.axis("off")

        plt.subplot(1, 4, 3)
        plt.imshow(overlay)
        title = f"Pred: {pred_label}\nConf: {confidence:.2f}"
        if true_label is not None:
            title = f"True: {true_label}\n" + title
        plt.title(title)
        plt.axis("off")

        plt.subplot(1, 4, 4)
        sns.barplot(data=result_df, x="probability", y="class")
        plt.xlim(0, 1)
        plt.title("Probabilites")

        plt.tight_layout()
        plt.show()

    return {
        "image_path": image_path,
        "true_label": true_label,
        "pred_label": pred_label,
        "confidence": confidence,
        "probabilities": result_df,
        "heatmap": heatmap_resized,
        "overlay": overlay,
    }

example_row = test_df.sample(1, random_state=RANDOM_STATE).iloc[0]
_ = predict_and_gradcam(example_row["image_path"], true_label=example_row["label"])


### Interpretation

Cette figure montre quatre elements: image pretraitee, heatmap Grad-CAM, superposition heatmap + IRM, et probabilites par classe. Si la heatmap se concentre sur le cerveau plutot que sur le fond noir, c'est un bon signe.


## 8. Grad-CAM par classe


In [ ]:
plt.figure(figsize=(12, 4 * len(classes)))

for class_idx, class_name in enumerate(classes):
    row = test_df[test_df["label"] == class_name].sample(1, random_state=RANDOM_STATE).iloc[0]
    img_array = load_preprocess_image(row["image_path"])
    input_batch = img_array[np.newaxis, ...]
    proba = model.predict(input_batch, verbose=0)[0]
    pred_idx = int(np.argmax(proba))
    pred_label = index_to_class[pred_idx]
    heatmap = make_gradcam_heatmap(input_batch, model, last_conv_layer_name, pred_index=pred_idx)
    overlay, _ = overlay_heatmap_on_gray(img_array, heatmap)

    plt.subplot(len(classes), 3, class_idx * 3 + 1)
    plt.imshow(img_array.squeeze(), cmap="gray")
    plt.title(f"True: {class_name}")
    plt.axis("off")

    plt.subplot(len(classes), 3, class_idx * 3 + 2)
    plt.imshow(heatmap, cmap="jet")
    plt.title("Grad-CAM")
    plt.axis("off")

    plt.subplot(len(classes), 3, class_idx * 3 + 3)
    plt.imshow(overlay)
    plt.title(f"Pred: {pred_label}\nConf: {proba[pred_idx]:.2f}")
    plt.axis("off")

plt.suptitle("Grad-CAM par classe", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()


### Interpretation

Cette grille permet de comparer les regions utilisees par le modele pour chaque classe. On cherche des zones anatomiquement plausibles: regions internes, ventricules, contours cerebraux, espaces sombres.


## 9. Comparer predictions correctes et incorrectes


In [ ]:
y_true = []
y_pred = []
y_conf = []

for _, row in test_df.iterrows():
    arr = load_preprocess_image(row["image_path"])
    proba = model.predict(arr[np.newaxis, ...], verbose=0)[0]
    pred_idx = int(np.argmax(proba))
    y_true.append(row["label"])
    y_pred.append(index_to_class[pred_idx])
    y_conf.append(float(proba[pred_idx]))

pred_df = test_df.copy()
pred_df["pred_label"] = y_pred
pred_df["confidence"] = y_conf
pred_df["correct"] = pred_df["label"] == pred_df["pred_label"]

display(pred_df[["label", "pred_label", "confidence", "correct"]].head())
print("Correct:", pred_df["correct"].sum())
print("Incorrect:", (~pred_df["correct"]).sum())


### Interpretation

On separe les predictions correctes et incorrectes. Les erreurs sont utiles pour comprendre les limites du modele et les confusions entre stades proches.


## 10. Grad-CAM sur erreurs du modele


In [ ]:
wrong_df = pred_df[~pred_df["correct"]]

if len(wrong_df) == 0:
    print("Aucune erreur trouvee dans le test.")
else:
    sample_wrong = wrong_df.sample(min(6, len(wrong_df)), random_state=RANDOM_STATE)
    for _, row in sample_wrong.iterrows():
        _ = predict_and_gradcam(row["image_path"], true_label=row["label"], show=True)


### Interpretation

Les Grad-CAM sur les erreurs aident a comprendre pourquoi le modele se trompe. Si la heatmap est sur le fond ou une zone non informative, le modele peut etre influence par un artefact.


## 11. Grad-CAM sur test augmente de robustesse


In [ ]:
if test_augmented_df is None:
    print("Pas de test augmente disponible.")
else:
    sample_aug = test_augmented_df.sample(min(4, len(test_augmented_df)), random_state=RANDOM_STATE)
    for _, row in sample_aug.iterrows():
        _ = predict_and_gradcam(row["image_path"], true_label=row["label"], show=True)


### Interpretation

Cette cellule verifie si le modele reste coherent sur des images legerement augmentees. Si la prediction change trop facilement, le modele manque de robustesse.


# Conclusion de l'etape 4

Grad-CAM apporte une explication visuelle aux predictions du CNN.

Dans un projet medical, cette etape est importante parce qu'elle permet de verifier si le modele s'appuie sur des zones plausibles de l'IRM plutot que sur des artefacts.

Les heatmaps doivent etre interpretees avec prudence: elles indiquent les zones importantes pour le modele, mais elles ne remplacent pas une segmentation anatomique ni un diagnostic medical.

Etape suivante: creation d'une API et d'une application frontend pour uploader une IRM et afficher la prediction avec les probabilites.
